# Dasbor Monitoring Busway JakCCTV di Kaggle (GPU)

> **Catatan bahasa:** teks bahasa Inggris ada di bawah teks bahasa Indonesia. *(Language note: the English text is right below the Indonesian text.)*

Menjalankan dasbor (live 4 kamera, log kendaraan, peta, insight) di GPU Kaggle, lalu membuat **link publik** lewat Cloudflare Tunnel supaya siapa pun bisa membukanya dari browser masing-masing.

Kode: https://github.com/Paradiess-svg/jakcctv-busway-monitor Â· Dataset: https://www.kaggle.com/datasets/izdiharwibowo/jakcctv-busway-tj-vs-non-tj-yolo

## Cara termudah (sumber: GitHub, default)
Settings: **Accelerator GPU T4**, **Internet On**, lalu **Run All**. Tidak perlu dataset. Update kode = push ke GitHub, lalu jalankan ulang notebook.

## Alternatif (sumber: dataset, set `SUMBER = 'dataset'` di sel 1)
1. Di laptop: `python src/siapkan_dasbor_cloud.py` â†’ `outputs/dasbor_cloud.zip`.
2. Kaggle â†’ Create â†’ New Dataset â†’ upload `dasbor_cloud.zip`, judul `jakcctv-dasbor`.
3. Di notebook ini: Add Input â†’ dataset `jakcctv-dasbor`. Settings: **GPU T4**, **Internet On**.

## Menjalankan
Jalankan sel 1â€“5 berurutan (boleh Run All). Link muncul di sel 5.
- **Link penonton** â†’ bagikan ke teman, hanya bisa melihat.
- **Link admin** â†’ untuk kamu saja: on/off kamera, verifikasi deteksi âœ“/âœ—, upload video. **Jangan dibagikan.**
- Tambah `?lang=en` untuk tampilan bahasa Inggris, `?mode=busway` untuk studi kasus *Busway Only*.

## Catatan penting
- Biarkan tab notebook ini tetap terbuka. Sesi interaktif Kaggle berhenti kalau lama tidak aktif dan maksimal Â±12 jam; kalau sesi berhenti, link ikut mati.
- Link `trycloudflare.com` berubah setiap kali dijalankan ulang.
- Log dan snapshot tersimpan di `/kaggle/working/app/outputs/monitor`. Unduh lewat sel 7 sebelum sesi berakhir.
- Komentar kode: bahasa Indonesia dulu, bahasa Inggris di baris bawahnya. Pesan konsol memakai label netral (`VIEWER LINK`, `ADMIN LINK`).
- Sumber kebenaran notebook = `notebooks/dasbor_kaggle.ipynb` di repo GitHub. Kalau perlu diubah, ubah di repo lalu salin ke Kaggle (File â†’ Import Notebook).

---

# JakCCTV Busway Monitor: dashboard on Kaggle (GPU)

> **Language note:** the English text is below the Indonesian text.

Runs the dashboard (4 live cameras, vehicle log, map, insights) on a Kaggle GPU and creates a **public link** through a Cloudflare Tunnel, so anyone can open it in their own browser.

Code: https://github.com/Paradiess-svg/jakcctv-busway-monitor Â· Dataset: https://www.kaggle.com/datasets/izdiharwibowo/jakcctv-busway-tj-vs-non-tj-yolo

## Quick start (source: GitHub, default)
Settings: **Accelerator GPU T4**, **Internet On**, then **Run All**. No dataset input needed. To update the code, push to GitHub and run this notebook again.

## Alternative (source: dataset, set `SUMBER = 'dataset'` in cell 1)
1. On your laptop: `python src/siapkan_dasbor_cloud.py` â†’ `outputs/dasbor_cloud.zip`.
2. Kaggle â†’ Create â†’ New Dataset â†’ upload `dasbor_cloud.zip`, title `jakcctv-dasbor`.
3. In this notebook: Add Input â†’ dataset `jakcctv-dasbor`. Settings: **GPU T4**, **Internet On**.

## Running
Run cells 1â€“5 in order (Run All is fine). The links appear in cell 5.
- **Viewer link** â†’ share it; it can only view.
- **Admin link** â†’ for you only: switch cameras on/off, verify detections âœ“/âœ—, upload video. **Do not share.**
- Add `?lang=en` for the English interface, `?mode=busway` for the *Busway Only* case study.

## Notes
- Keep this notebook tab open. Interactive Kaggle sessions stop when idle and last at most ~12 hours; when the session stops, the link dies too.
- The `trycloudflare.com` link changes on every run.
- Logs and snapshots are stored in `/kaggle/working/app/outputs/monitor`. Download them with cell 7 before the session ends.
- Code comments are Indonesian first, English on the line below. Console messages use neutral labels (`VIEWER LINK`, `ADMIN LINK`).
- The source of truth for this notebook is `notebooks/dasbor_kaggle.ipynb` in the GitHub repo. Edit it there, then import it into Kaggle (File â†’ Import Notebook).


In [ ]:
# 1) Ambil aplikasi ke /kaggle/working/app
# 1) Get the app into /kaggle/working/app
#    SUMBER = 'github'  -> kode terbaru langsung dari GitHub (update cukup push ke repo)
#    SUMBER = 'dataset' -> dari input dataset jakcctv-dasbor (dasbor_cloud.zip)
#    SUMBER = 'github'  -> latest code straight from GitHub (update by pushing to the repo)
#    SUMBER = 'dataset' -> from the jakcctv-dasbor input dataset (dasbor_cloud.zip)
SUMBER = 'github'
REPO = 'https://github.com/Paradiess-svg/jakcctv-busway-monitor.git'
import os, shutil, glob, subprocess, sys, time, secrets, re, json, urllib.request
APP = '/kaggle/working/app'
if os.path.exists(APP):
    shutil.rmtree(APP)
if SUMBER == 'github':
    subprocess.run(['git', 'clone', '--depth', '1', REPO, APP], check=True)
else:
    kandidat = glob.glob('/kaggle/input/**/jakcctv_app/dashboard/server.py', recursive=True)
    if not kandidat:   # zip belum diekstrak otomatis / zip not auto-extracted
        z = glob.glob('/kaggle/input/**/dasbor_cloud.zip', recursive=True)
        assert z, 'Dataset jakcctv-dasbor belum ditambahkan (Add Input), atau pakai SUMBER = github / dataset jakcctv-dasbor not added, or use SUMBER = github'
        shutil.unpack_archive(z[0], '/kaggle/working/_ekstrak')
        kandidat = glob.glob('/kaggle/working/_ekstrak/**/jakcctv_app/dashboard/server.py', recursive=True)
    assert len(kandidat) == 1, kandidat
    shutil.copytree(os.path.dirname(os.path.dirname(kandidat[0])), APP)
print('App:', APP, f'(source: {SUMBER})'); print(sorted(os.listdir(APP)))

In [ ]:
# 2) Pasang library (torch & ffmpeg biasanya sudah ada di Kaggle)
# 2) Install libraries (torch & ffmpeg are usually preinstalled on Kaggle)
!pip install -q ultralytics fastapi "uvicorn[standard]" python-multipart lap
!which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)
import torch; print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
# 3) Tes: apakah server Kaggle bisa membuka stream JakCCTV? (server Kaggle ada di luar negeri)
# 3) Test: can the Kaggle server open the JakCCTV streams? (Kaggle servers are outside Indonesia)
import yaml
cfg = yaml.safe_load(open(f'{APP}/config/config.yaml'))
kam = cfg['dasbor']['kamera_default']
sys.path.insert(0, f'{APP}/src')
from capture import FFMPEG_INPUT, configure_hls_opts   # opsi HLS sesuai versi ffmpeg / HLS options for this ffmpeg version
configure_hls_opts('ffmpeg')
for k in kam:
    url = cfg['cameras'][k]['url']
    r = subprocess.run(['ffmpeg', *FFMPEG_INPUT, '-i', url,
                        '-frames:v', '1', '-f', 'null', '-'], capture_output=True, text=True, timeout=60)
    print(f"{k:22s}", 'OK' if r.returncode == 0 else 'FAILED: ' + r.stderr.strip()[-200:])
# Kalau semua GAGAL: stream JakCCTV kemungkinan menolak akses dari luar Indonesia -> pakai mode laptop.
# If all FAILED: JakCCTV probably blocks access from outside Indonesia -> run locally instead.

In [ ]:
# 4) Jalankan server dasbor di latar belakang (token admin dibuat acak)
# 4) Start the dashboard server in the background (random admin token)
TOKEN = secrets.token_urlsafe(8)
env = dict(os.environ, TJ_TOKEN=TOKEN, PYTHONUNBUFFERED='1')
log_srv = open('/kaggle/working/server.log', 'w')
srv = subprocess.Popen([sys.executable, 'dashboard/server.py', '--port', '8000'], cwd=APP, env=env,
                       stdout=log_srv, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(urllib.request.urlopen('http://127.0.0.1:8000/api/info', timeout=2).read().decode()); break
    except Exception:
        time.sleep(2)
else:
    print(open('/kaggle/working/server.log').read()[-3000:])

In [ ]:
# 5) Link publik lewat Cloudflare Tunnel (gratis, tanpa akun)
# 5) Public link through Cloudflare Tunnel (free, no account)
if not os.path.exists('/kaggle/working/cloudflared'):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
                               '/kaggle/working/cloudflared')
    os.chmod('/kaggle/working/cloudflared', 0o755)
log_cf = open('/kaggle/working/cloudflared.log', 'w')
cf = subprocess.Popen(['/kaggle/working/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'],
                      stdout=log_cf, stderr=subprocess.STDOUT)
LINK = None
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/kaggle/working/cloudflared.log').read())
    if m:
        LINK = m.group(0); break
    time.sleep(2)
print('VIEWER LINK:', LINK)
print('ADMIN LINK :', f'{LINK}/?admin={TOKEN}   <- DO NOT SHARE')
print('ENGLISH UI :', f'{LINK}/?lang=en')
# Live otomatis menyala saat link pertama kali dibuka.
# Live cameras start automatically when the link is first opened.

In [ ]:
# 6) Pantau (opsional): status kamera tiap 60 detik. Hentikan sel ini kapan saja (server tetap jalan).
# 6) Monitor (optional): camera status every 60 s. Stop this cell any time (the server keeps running).
try:
    while True:
        s = json.loads(urllib.request.urlopen('http://127.0.0.1:8000/api/live/status').read())
        print(time.strftime('%H:%M:%S'), ' | '.join(f"{k['kamera']}: {k['stream']} {k.get('fps_analisis', 0)} fps TJ {k['hitung']['tj']} non {k['hitung']['non_tj']}" for k in s['kamera']))
        time.sleep(60)
except KeyboardInterrupt:
    pass

In [ ]:
# 7) Simpan log & snapshot (zip) -> unduh dari tab Output, lalu matikan semuanya
# 7) Save logs & snapshots (zip) -> download from the Output tab, then shut everything down
shutil.make_archive('/kaggle/working/hasil_dasbor', 'zip', f'{APP}/outputs/monitor')
print('ZIP /kaggle/working/hasil_dasbor.zip -> Output tab')
try:
    urllib.request.urlopen(urllib.request.Request('http://127.0.0.1:8000/api/live/stop', method='POST', headers={'X-Token': TOKEN}))
except Exception as e:
    print(e)
for p in (cf, srv):
    p.terminate()
print('Server & tunnel stopped')